# Chapter1

```shell
pip install requests tavily-python openai python-dotenv
```

## Key Notes
### 智能体的类型
```mermaid
mindmap
  智能体的类型
    维度一：内部决策架构
      反应式：感知即行动
      模型式：引入内部模型
      基于目标和基于效用：规划和优化
    维度二：时间与反应性
      反应式智能体
        速度快、计算开销低
        缺乏规划、易局部最优
        例：安全气囊、高频交易
      规划式智能体
        深思熟虑、有战略远见
        时间与算力成本高
        例：AlphaGo、长途旅行规划
      混合式智能体
        分层设计：快反应＋慢规划
        LLM智能体：思考-行动-观察循环
    维度三：知识表示
      亚符号主义（连接主义）
        神经网络，从海量数据中学习
        强模式识别，但是黑箱
        类比：卡尼曼系统1，快思考
      符号主义
        规则库、知识图谱、逻辑推理
        可解释，但脆弱、知识获取瓶颈
        类比：卡尼曼系统2，慢思考
      神经符号主义
        融合模式识别与逻辑推理
        例：LLM驱动的智能体
```

### 智能体的构成与运行原理
#### 任务环境定义：PEAS模型
PEAS：性能度量（Performance）、环境（Environment）、执行器（Actuators）、传感器（Sensors）

#### 智能体的运行机制
```mermaid
flowchart LR
    E[环境 Environment]

    subgraph AGENT[智能体 Agent]
        direction LR
        P[感知 Perception]
        subgraph T[思考 Thought]
            direction LR
            PL[规划 Planning]
            TS[工具选择 Tool Selection]
            PL --> TS
        end
        A[行动 Action]
        P --> PL
        TS --> A
    end

    E -- 观察 Observation --> P
    A -- 状态变化 --> E
```


## Sample

### System prompt

In [13]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一：
1. 调用工具：function_name(arg_name="arg_value")
2. 结束任务：Finish[最终答案]

# 重要提示:
- 每次只输出一对Thought-Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[最终答案] 格式结束

请开始吧！
"""

### 查询天气

In [ ]:

import requests


def get_weather(city):
    """
    通过调用 wttr.in API 查询真实的天气信息。
    """
    # API端点，我们请求JSON格式的数据
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        # 发起网络请求
        response = requests.get(url)
        # 检查响应状态码是否为200 (成功)
        response.raise_for_status() 
        # 解析返回的JSON数据
        data = response.json()
        
        # 提取当前天气状况
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        
        # 格式化成自然语言返回
        return f"{city}当前天气:{weather_desc}，气温{temp_c}摄氏度"
        
    except requests.exceptions.RequestException as e:
        # 处理网络错误
        return f"错误:查询天气时遇到网络问题 - {e}"
    except (KeyError, IndexError) as e:
        # 处理数据解析错误
        return f"错误:解析天气数据失败，可能是城市名称无效 - {e}"

### 搜索并推荐旅游景点

In [15]:
import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    
    """
    根据城市和天气推荐旅游景点。
    """

    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未设置TAVILY_API_KEY环境变量。请在系统环境变量中添加TAVILY_API_KEY。"

    # 初始化 Tavily API 客户端
    tavily_client = TavilyClient(api_key)

    query = f"请根据{city}的天气情况'{weather}'，推荐适合的旅游景点。"

    try:
        # 调用 Tavily API 获取推荐景点
        response = tavily_client.search(query=query, search_depth="basic", include_answers=True)

        if response.get("answer"):
            # 提取推荐景点信息
            attractions = response["answer"]
            return f"{city}的天气情况为'{weather}'，推荐的旅游景点有: {attractions}"

        formatted_result = []
        for result in response.get("results", []):
            formatted_result.append(f"-{result['title']}: {result['content']}")

        if not formatted_result:
            return f"未找到适合的旅游景点推荐。"

        return f"{city}的天气情况为'{weather}'，推荐的旅游景点有:\n" + "\n".join(formatted_result)

    except Exception as e:
        return f"错误:调用Tavily API时发生异常 - {e}"

    

### 定义工具

In [16]:
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}

### 接入大语言模型

In [17]:
from openai import OpenAI

class OpenAICompatibleClient:
    """
    一个用于调用任何兼容OpenAI接口的LLM服务的客户端    
    """
    def __init__(self, api_key: str, base_url: str, model_name: str):
        self.model_name = model_name
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate_response(self, messages: list) -> str:
        """
        传入完整对话历史（含system提示词），返回模型回复文本。
        多轮循环时由调用方负责把每轮的回复和工具结果追加进messages。
        """
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=messages,
                stream=False
            )
            answer = response.choices[0].message.content
            if answer is None:
                return "错误:模型未返回文本内容（可能是触发了内容过滤）"
            return answer
        except Exception as e:
            return f"错误:调用大语言模型时发生异常 - {e}"

### 运行示例

In [18]:
import os
import re
from dotenv import load_dotenv

# --- 1. 配置LLM客户端（从 .env 文件读取密钥）---
load_dotenv(override=True)

API_KEY = os.environ["OPENAI_API_KEY"]
BASE_URL = os.environ["OPENAI_BASE_URL"]
MODEL_NAME = os.environ["MODEL_NAME"]
# TAVILY_API_KEY 已由 load_dotenv() 写入 os.environ，get_attraction 可直接读取

llm = OpenAICompatibleClient(
    api_key=API_KEY,
    base_url=BASE_URL,
    model_name=MODEL_NAME
)

# --- 2. 初始化 ---
user_prompt = "你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。"
prompt_history = [f"用户请求: {user_prompt}"]

print(f"用户输入: {user_prompt}\n" + "="*40)

# --- 3. 运行主循环 ---
for i in range(5): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)
    
    # 3.2. 调用LLM进行思考
    llm_output = llm.generate_response([
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": full_prompt},
    ])
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != llm_output.strip():
            llm_output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{llm_output}\n")
    prompt_history.append(llm_output)
    
    # 3.3. 解析并执行行动
    action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
    if not action_match:
        observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue
    action_str = action_match.group(1).strip()

    if action_str.startswith("Finish"):
        final_answer = re.match(r"Finish\[(.*)\]", action_str).group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break
    
    tool_name = re.search(r"(\w+)\(", action_str).group(1)
    args_str = re.search(r"\((.*)\)", action_str).group(1)
    kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))

    if tool_name in available_tools:
        observation = available_tools[tool_name](**kwargs)
    else:
        observation = f"错误:未定义的工具 '{tool_name}'"

    # 3.4. 记录观察结果
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)

用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。
--- 循环 1 ---

模型输出:
Thought: 首先需要查询北京的天气情况，以便根据具体天气推荐合适的景点。
Action: get_weather(city="北京")

Observation: 北京当前天气:Smoky haze，气温26摄氏度
--- 循环 2 ---

模型输出:
Thought: 已经获取到北京当前天气为Smoky haze（雾霾/烟霾），接下来需要根据北京和当前天气查询推荐的旅游景点。
Action: get_attraction(city="北京", weather="Smoky haze")

Observation: 北京的天气情况为'Smoky haze'，推荐的旅游景点有:
-Beijing天气状况：气温| 30天预报 - AQI.in: sunny

体感温度 24°C

Chances of Rain1%

25AQI

空气清新.

湿度

41%

预计湿度舒适.

sunny
sunny
sunny
sunny
sunny
sunny
sunny
sunny
smoky haze
smoky haze
smoky haze
smoky haze
smoky haze
smoky haze
clear
smoky haze
smoky haze
smoky haze
smoky haze
smoky haze
smoky haze
smoky haze
smoky haze
smoky haze [...] 可能晴天；防晒保护可能有帮助.

最后更新：2026-09-01 10:00 (本地时间)

## Beijing 天气参数

32°NNE

风向

13.7km/h

风速

Gust Speed Image

阵风速度

5.4m/s

云层覆盖

0%

能见度

10km

降水

降水

0mm

当前的降水概率为 0mm

气压

1018mb

当前的气压水平为 1018mb。

紫外线指数

4.7适中

当前的紫外线指数为 4.7，请参考相关建议！

## China 的热门趋势

China 的热门趋势

What’s the CO₂ Like Inside a Delhi Metro? 🚇

Wh

## Practice

1. 请分析以下四个 case 中的主体是否属于智能体，如果是，那么属于哪种类型的智能体（可以从多个分类维度进行分析），并说明理由：

   case A：一台符合冯·诺依曼结构的超级计算机，拥有高达每秒 2EFlop 的峰值算力

   否

   case B：特斯拉自动驾驶系统在高速公路上行驶时，突然检测到前方有障碍物，需要在毫秒级做出刹车或变道决策

   是，属于混合式智能体。理由：特斯拉自动驾驶系统能够感知环境（检测障碍物）、做出决策（刹车或变道）并执行动作，具备自主性和适应性。

   case C：AlphaGo在与人类棋手对弈时，需要评估当前局面并规划未来数十步的最优策略

   是，属于规划式智能体。理由：AlphaGo需要进行深度思考和策略规划，以找到最优的下一步棋。

   case D：ChatGPT 扮演的智能客服在处理用户投诉时，需要查询订单信息、分析问题原因、提供解决方案并安抚用户情绪

   是，属于反应式智能体。理由：ChatGPT能够根据用户的输入快速响应，提供即时的帮助和解决方案。

2. 假设你需要为一个"智能健身教练"设计任务环境。这个智能体能够：
   - 通过可穿戴设备监测用户的心率、运动强度等生理数据
   - 根据用户的健身目标（减脂/增肌/提升耐力）动态调整训练计划
   - 在用户运动过程中提供实时语音指导和动作纠正
   - 评估训练效果并给出饮食建议

   请使用 PEAS 模型完整描述这个智能体的任务环境，并分析该环境具有哪些特性（如部分可观察、随机性、动态性等）。

   PEAS 模型分析：
   - **Performance Measure (性能度量)**: 用户的健身效果、训练满意度、目标达成率等。
   - **Environment (环境)**: 用户的运动场所、可穿戴设备、饮食环境等。
   - **Actuators (执行器)**: 语音输出设备、训练计划调整功能等。
   - **Sensors (传感器)**: 可穿戴设备的生理数据监测功能等。

   该环境具有以下特性：
   - **部分可观察**: 智能体无法完全感知用户的所有状态，只能通过传感器获取部分信息。
   - **随机性**: 用户的运动表现和生理反应具有一定的随机性。
   - **动态性**: 用户的健身目标和环境条件会随时间变化，智能体需要不断适应。

3. 某电商公司正在考虑两种方案来处理售后退款申请：
   
   方案 A（`Workflow`）：设计一套固定流程，例如：

   A.1 对于一般商品且在 7 天之内，金额 `< 100RMB` 自动通过；`100-500RMB `由客服审核；`>500RMB` 需主管审批；而特殊商品（如定制品）一律拒绝退款
   
   A.2 对于超过 7 天的商品，无论金额，只能由客服审核或主管审批；
   
   方案 B（`Agent`）：搭建一个智能体系统，让它理解退款政策、分析用户历史行为、评估商品状况，并自主决策是否批准退款
   
   请分析：
   - 这两种方案各自的优缺点是什么？
   - 在什么情况下 `Workflow` 更合适？什么情况下 `Agent` 更有优势？如果你是该电商公司的负责人，你更倾向于采用哪种方案？
   - 是否存在一个方案 C，能够结合两种方案，达到扬长避短的效果？
  
  Q1. 方案 A 的优点是流程明确、易于管理和执行，缺点是缺乏灵活性，无法处理复杂或特殊情况。方案 B 的优点是能够自主决策、适应不同情况，缺点是需要较高的技术投入和维护成本。

  Q2. 当退款申请数量较少且规则明确时，`Workflow` 更合适；当退款申请数量大且情况复杂多变时，`Agent` 更有优势。如果我是该电商公司的负责人，我会倾向于采用方案 B，因为它能够提高处理效率和用户满意度。

  Q3. 方案 C 可以结合两种方案的优点，例如在大部分情况下使用固定流程处理退款申请，而对于特殊或复杂情况，智能体可以介入进行自主决策，从而达到扬长避短的效果。

4. 在 1.3 节的智能旅行助手基础上，请思考如何添加以下功能（可以只描述设计思路，也可以进一步尝试代码实现）：

   > <strong>提示</strong>：思考如何修改 `Thought-Action-Observation` 循环来实现这些功能。

   - 添加一个"记忆"功能，让智能体记住用户的偏好（如喜欢历史文化景点、预算范围等）
   - 当推荐的景点门票已售罄时，智能体能够自动推荐备选方案
   - 如果用户连续拒绝了 3 个推荐，智能体能够反思并调整推荐策略

5. 卡尼曼的"系统 1"（快速直觉）和"系统 2"（慢速推理）理论<sup>[2]</sup>为神经符号主义 AI 提供了很好的类比。请首先构思一个具体的智能体的落地应用场景，然后说明场景中的：

   > <strong>提示</strong>：医疗诊断助手、法律咨询机器人、金融风控系统等都是常见的应用场景

   - 哪些任务应该由"系统 1"处理？
   - 哪些任务应该由"系统 2"处理？
   - 这两个系统如何协同工作以达成最终目标？
  

6. 尽管大语言模型驱动的智能体系统展现出了强大的能力，但它们仍然存在诸多局限。请分析以下问题：
   - 为什么智能体或智能体系统有时会产生"幻觉"（生成看似合理但实际错误的信息）？
   - 在 1.3 节的案例中，我们设置了最大循环次数为 5 次。如果没有这个限制，智能体可能会陷入什么问题？
   - 如何评估一个智能体的"智能"程度？仅使用准确率指标是否足够？
   
   Q1. 智能体产生"幻觉"的原因主要包括：训练数据的偏差、模型对上下文理解的局限性、缺乏事实验证机制等。大语言模型在生成文本时，可能会基于模式匹配而非真实信息，从而导致错误输出。

   Q2. 如果没有最大循环次数的限制，智能体可能会陷入无限循环，反复尝试相同的操作而无法收敛到有效的解决方案。这可能导致资源浪费和性能下降。

   Q3. 评估智能体的"智能"程度需要综合考虑多个指标，包括准确率、响应速度、适应性、用户满意度等。仅使用准确率指标是不够的，因为它无法全面反映智能体在实际应用中的表现和用户体验。